# MiniMax H3 Ref2VA LoRA training

This notebook trains the MiniMax H3 Ref2VA transformer with Ostris AI Toolkit. It uses still photos as one-frame targets and pairs each target with a same-basename reference photo. The Qwen3-VL image encoder and H3 visual VAE are part of the H3 pipeline; this notebook does not train Qwen Image.

The AI Toolkit Ref2VA example configuration is marked unverified upstream. This notebook pins the inspected toolkit commit and derives the configuration from its documented Ref2VA fields. Installation and packaging can be checked locally; remote training still requires an appropriate Colab GPU and has not been run by this repository publisher.

The H3 model files and the AI Toolkit training adapter are downloaded at runtime. Review and accept their licenses before training.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("A Colab GPU runtime is required.")

gpu_name = torch.cuda.get_device_name(0)
vram_gib = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
disk_free_gib = round(shutil.disk_usage("/content").free / (1024 ** 3), 2)
print("GPU:", gpu_name)
print("VRAM:", vram_gib, "GiB")
print("Free disk:", disk_free_gib, "GiB")
if disk_free_gib < 60:
    raise RuntimeError("H3 Ref2VA needs at least 60 GiB free disk for model files, toolkit, and outputs.")
if vram_gib < 48:
    print("Warning: no official minimum VRAM is published; this runtime may run out of memory.")


In [ ]:
from google.colab import drive

LORA_NAME = os.environ.get("LORA_NAME", "character").strip()
RUN_ID = os.environ.get("LORA_RUN_ID", time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()))
TRIGGER_WORD = os.environ.get("TRIGGER_WORD", f"{LORA_NAME} person").strip()
DRIVE_SUBPATH = Path(os.environ.get("DRIVE_SUBPATH", f"MiniMax_H3_LoRAs/{LORA_NAME}"))
if DRIVE_SUBPATH.is_absolute() or ".." in DRIVE_SUBPATH.parts:
    raise ValueError("DRIVE_SUBPATH must be relative to MyDrive and must not contain '..'.")

DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_MYDRIVE = DRIVE_MOUNT_POINT / "MyDrive"
if not DRIVE_MYDRIVE.is_dir():
    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
if not DRIVE_MYDRIVE.is_dir():
    raise RuntimeError("Google Drive did not mount; stopping before training.")

DRIVE_TARGET_DIR = (DRIVE_MYDRIVE / DRIVE_SUBPATH).resolve()
if DRIVE_MYDRIVE.resolve() not in DRIVE_TARGET_DIR.parents:
    raise ValueError("DRIVE_SUBPATH resolves outside MyDrive.")
DRIVE_TARGET_DIR.mkdir(parents=True, exist_ok=True)
RUN_DRIVE_DIR = DRIVE_TARGET_DIR / RUN_ID
RUN_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ZIP = Path("/content/dataset.zip")
DATASET_DIR = Path("/content/h3_ref2va_dataset")
if not DATASET_ZIP.is_file():
    raise FileNotFoundError(f"Dataset archive is missing: {DATASET_ZIP}")
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)
with zipfile.ZipFile(DATASET_ZIP) as archive:
    archive.extractall(DATASET_DIR)

TARGETS_DIR = DATASET_DIR / "targets"
REFERENCES_DIR = DATASET_DIR / "references"
VALID_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
target_images = sorted(
    p for p in TARGETS_DIR.glob("*")
    if p.is_file() and p.suffix.lower() in VALID_EXTS
)
reference_images = sorted(
    p for p in REFERENCES_DIR.glob("*")
    if p.is_file() and p.suffix.lower() in VALID_EXTS
)
reference_stems = {p.stem for p in reference_images}
if not target_images or not reference_images:
    raise ValueError("Dataset must contain target and reference images.")
for image in target_images:
    if image.stem not in reference_stems:
        raise ValueError(f"No same-basename Ref2VA reference for {image.name}.")
    caption = image.with_suffix(".txt")
    if not caption.is_file() or not caption.read_text(encoding="utf-8").strip():
        raise ValueError(f"Missing non-empty caption: {caption}")

sample_reference = next(p for p in reference_images if p.stem == target_images[0].stem)
print("Validated image targets:", len(target_images))
print("Validated reference images:", len(reference_images))
print("Trigger token:", TRIGGER_WORD)


In [ ]:
AI_TOOLKIT_COMMIT = "ecee894ed2b1f3716d9d7326693061ec1a3105bb"
AI_TOOLKIT_URL = "https://github.com/ostris/ai-toolkit.git"
AI_TOOLKIT_DIR = Path("/content/ai-toolkit")

if not (AI_TOOLKIT_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", AI_TOOLKIT_URL, str(AI_TOOLKIT_DIR)],
        check=True,
    )
subprocess.run(
    ["git", "fetch", "--depth", "1", "origin", AI_TOOLKIT_COMMIT],
    cwd=AI_TOOLKIT_DIR,
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", "FETCH_HEAD"],
    cwd=AI_TOOLKIT_DIR,
    check=True,
)
subprocess.run(
    ["git", "submodule", "update", "--init", "--recursive"],
    cwd=AI_TOOLKIT_DIR,
    check=True,
)

SETUP_MARKER = Path(f"/content/.ai_toolkit_{AI_TOOLKIT_COMMIT[:12]}_requirements")
if not SETUP_MARKER.is_file():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        cwd=AI_TOOLKIT_DIR,
        check=True,
    )
    SETUP_MARKER.touch()
print("AI Toolkit pinned to:", AI_TOOLKIT_COMMIT)

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)


In [ ]:
import yaml

RANK = int(os.environ.get("LORA_RANK", "16"))
ALPHA = int(os.environ.get("LORA_ALPHA", "16"))
STEPS = int(os.environ.get("LORA_STEPS", "1000"))
LR = float(os.environ.get("LORA_LR", "1e-4"))
SAVE_EVERY = min(250, STEPS)
OUTPUT_ROOT = Path("/content/output")
RUN_OUTPUT_DIR = OUTPUT_ROOT / f"{LORA_NAME}_{RUN_ID}"
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

config_data = {
    "job": "extension",
    "config": {
        "name": f"{LORA_NAME}_{RUN_ID}",
        "process": [{
            "type": "sd_trainer",
            "training_folder": str(RUN_OUTPUT_DIR),
            "device": "cuda:0",
            "trigger_word": TRIGGER_WORD,
            "network": {
                "type": "lora",
                "linear": RANK,
                "linear_alpha": ALPHA,
                "network_kwargs": {"ignore_if_contains": ["adaln_proj"]},
            },
            "save": {
                "dtype": "float16",
                "save_every": SAVE_EVERY,
                "max_step_saves_to_keep": 4,
                "push_to_hub": False,
            },
            "datasets": [{
                "folder_path": str(TARGETS_DIR),
                "control_path": str(REFERENCES_DIR),
                "caption_ext": "txt",
                "caption_dropout_rate": 0.05,
                "num_frames": 1,
                "fps": 24,
                "auto_frame_count": False,
                "do_audio": False,
                "cache_latents_to_disk": True,
                "resolution": [512, 768],
            }],
            "train": {
                "batch_size": 1,
                "steps": STEPS,
                "gradient_checkpointing": True,
                "noise_scheduler": "flowmatch",
                "timestep_type": "shift",
                "do_guidance_loss": True,
                "guidance_loss_target": 3.5,
                "audio_loss_multiplier": 0.0,
                "optimizer": "adamw8bit",
                "lr": LR,
                "dtype": "bf16",
                "cache_text_embeddings": True,
            },
            "model": {
                "name_or_path": "Comfy-Org/MiniMax-H3",
                "arch": "minimax_h3_ref2va",
                "quantize": True,
                "qtype": "convrot8",
                "quantize_te": True,
                "qtype_te": "nvfp4",
                "low_vram": True,
                "assistant_lora_path": (
                    "ostris/minimax_h3_training_adapter/"
                    "minimax_h3_ref2va_training_adapter_v1.safetensors"
                ),
                "model_kwargs": {
                    "partition": "ref2va_pruned",
                    "image_refs_as_video": False,
                },
            },
            "sample": {
                "sampler": "flowmatch",
                "sample_every": STEPS,
                "width": 512,
                "height": 512,
                "num_frames": 1,
                "fps": 24,
                "guidance_scale": 1,
                "sample_steps": 28,
                "samples": [{
                    "prompt": f"A clear portrait of {TRIGGER_WORD} from the reference image.",
                    "ctrl_img_1": str(sample_reference),
                }],
            },
        }],
    },
}
CONFIG_PATH = Path("/content/config.yaml")
CONFIG_PATH.write_text(yaml.safe_dump(config_data, sort_keys=False), encoding="utf-8")
shutil.copy2(CONFIG_PATH, RUN_DRIVE_DIR / "config.yaml")
print("Architecture: minimax_h3_ref2va")
print("Target images:", len(target_images), "| LoRA rank:", RANK, "| Steps:", STEPS)


In [ ]:
import subprocess
import time

train_command = [sys.executable, "run.py", str(CONFIG_PATH)]
print("Starting MiniMax H3 Ref2VA LoRA training.")
started_at = time.monotonic()
last_sync_at = 0.0
process = subprocess.Popen(
    train_command,
    cwd=AI_TOOLKIT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,
)
assert process.stdout is not None
try:
    for line in process.stdout:
        print(line, end="", flush=True)
        now = time.monotonic()
        if now - last_sync_at >= 30:
            last_sync_at = now
            for checkpoint in RUN_OUTPUT_DIR.rglob("*.safetensors"):
                try:
                    shutil.copy2(checkpoint, RUN_DRIVE_DIR / checkpoint.name)
                except OSError as exc:
                    print(f"Drive checkpoint sync warning: {exc}", flush=True)
    return_code = process.wait()
except BaseException:
    process.terminate()
    process.wait(timeout=20)
    raise

elapsed_seconds = round(time.monotonic() - started_at, 1)
print(f"Training finished in {elapsed_seconds}s (exit code {return_code}).")
if return_code != 0:
    raise RuntimeError(f"AI Toolkit training exited with status {return_code}.")


In [ ]:
produced_loras = sorted(
    RUN_OUTPUT_DIR.rglob("*.safetensors"),
    key=lambda path: path.stat().st_mtime,
)
if not produced_loras:
    raise FileNotFoundError(f"No H3 LoRA checkpoint was written under {RUN_OUTPUT_DIR}.")

final_lora = produced_loras[-1]
staged_lora = Path("/content/lora_output.safetensors")
shutil.copy2(final_lora, staged_lora)

drive_lora = RUN_DRIVE_DIR / f"{LORA_NAME}_minimax_h3_ref2va.safetensors"
shutil.copy2(final_lora, drive_lora)
for sample in RUN_OUTPUT_DIR.rglob("*"):
    if sample.is_file() and sample.suffix.lower() in {".png", ".jpg", ".mp4"}:
        shutil.copy2(sample, RUN_DRIVE_DIR / sample.name)

run_summary = {
    "status": "completed",
    "architecture": "minimax_h3_ref2va",
    "toolkit_commit": AI_TOOLKIT_COMMIT,
    "run_id": RUN_ID,
    "checkpoint": str(drive_lora),
    "target_images": len(target_images),
    "trigger": TRIGGER_WORD,
}
(RUN_DRIVE_DIR / "run_summary.json").write_text(
    json.dumps(run_summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("H3 Ref2VA LoRA:", staged_lora)
print("Google Drive archive:", RUN_DRIVE_DIR)
print("Checkpoint size MiB:", round(staged_lora.stat().st_size / (1024 ** 2), 2))
